# FT-900 训练期报告月份身份核查

在已收源字节上核对首页正式报告标题的月份和 08:30 发布标识，不补抓、不读市场结果。

In [1]:
from pathlib import Path
import hashlib,json,re,datetime,fitz
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023';plan=json.loads((card/'full_training_report_identity_plan.json').read_text());rows=json.loads((card/plan['parent_row_audit']).read_text())['rows']
assert len(rows)==plan['expected_months']==144
out=[]
for row in rows:
    month=row['reference_month'];name=row['source_url'].rsplit('/',1)[-1];path=root/'data/snapshots/census_ft900_training_20261023'/name;payload=path.read_bytes();assert hashlib.sha256(payload).hexdigest()==row['source_pdf_sha256'] and len(payload)==row['source_bytes']
    first=' '.join(fitz.open(stream=payload,filetype='pdf')[0].get_text().split())
    match=re.search(r'U\.S\. INTERNATIONAL TRADE IN GOODS AND SERVICES,?\s+([A-Za-z]+)\s+(\d{4})',first,re.I)
    expected=datetime.datetime.strptime(month,'%Y-%m').strftime('%B %Y')
    printed=(match.group(1)+' '+match.group(2)) if match else None
    identity=bool(printed and printed.lower()==expected.lower())
    time=bool(re.search(r'8:30\s*(?:A\.?M\.?|AM)',first,re.I))
    out.append({'reference_month':month,'source_sha256':row['source_pdf_sha256'],'expected_report_title_month':expected,'printed_report_title_month':printed,'title_month_matches':identity,'first_page_0830_release_indicated':time})
result={'status':'all_144_printed_report_months_match' if all(x['title_month_matches'] for x in out) else 'report_month_identity_exceptions','months':144,'matching_titles':sum(x['title_month_matches'] for x in out),'first_page_0830_headers':sum(x['first_page_0830_release_indicated'] for x in out),'exceptions':[x for x in out if not x['title_month_matches'] or not x['first_page_0830_release_indicated']],'historical_first_public_bytes_qualified':False,'classification_continuity_qualified':False,'Qlib_factor_rows':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'detail':out}
(card/'full_training_report_identity.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','matching_titles','first_page_0830_headers')},'exceptions',len(result['exceptions']))

{'status': 'report_month_identity_exceptions', 'matching_titles': 143, 'first_page_0830_headers': 144} exceptions 1


## 结论边界

报告标题与计划月份匹配只证明文件内容身份；历史首次公开字节、时序与 2012 分类仍待核实。

后续限定报表身份核查已完成：143 份标题直接匹配，2022-12 年终合并标题经单例合同核对，144 份报告月份均有来源证据；历史首次公开字节及跨 2012 分类可比性仍未获资格。